# Project 1 — Pre-training Colab starter

Environment check, tokenization, a from-scratch training loop, checkpointing, and sampling for the narrative base model. This is the supported starting point; the design choices in `docs/assignment_brief.md` are yours to make.

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())

PARAM_TARGET = 32_000_000
PARAM_MAX = 33_600_000

## 1. Data

Load the tokenized narrative corpus shard supplied with the starter repository.

In [ ]:
import numpy as np

def load_shard(path: str) -> np.ndarray:
    return np.fromfile(path, dtype=np.uint16)

# train_ids = load_shard('data/train.bin')
# val_ids = load_shard('data/val.bin')

## 2. Model

A minimal decoder-only Transformer block, sized to fit the parameter budget.

In [ ]:
import torch.nn as nn

class TinyDecoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 256, n_layer: int = 6, n_head: int = 4, block_size: int = 256):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_head, dim_feedforward=4 * d_model, batch_first=True)
        self.blocks = nn.TransformerEncoder(layer, num_layers=n_layer)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        b, t = idx.shape
        pos = torch.arange(t, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(t, device=idx.device)
        x = self.blocks(x, mask=mask, is_causal=True)
        return self.head(x)

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [ ]:
# Parameter preflight — run this before any costly training.
# model = TinyDecoder(vocab_size=8192)
# n = count_params(model)
# assert n <= PARAM_MAX, f'{n} exceeds the 33.6M eligibility boundary'
# print(f'{n:,} learned parameters')

## 3. Training loop, checkpointing, and sampling

A plain training step, checkpoint save, and greedy/temperature sampling for qualitative checks.

In [ ]:
def train_step(model, batch, optimizer):
    x, y = batch
    logits = model(x)
    loss = nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

# torch.save(model.state_dict(), 'checkpoint.pt')

In [ ]:
@torch.no_grad()
def sample(model, prefix_ids: torch.Tensor, max_new_tokens: int = 100, temperature: float = 0.8) -> torch.Tensor:
    ids = prefix_ids.clone()
    for _ in range(max_new_tokens):
        logits = model(ids)[:, -1, :] / temperature
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)
    return ids

## 4. Evaluation

Run the public evaluation pack's `metrics.py` against your checkpoint before submitting.